# Chapter 8: the 14 tables and the share check

Section 8.6 builds all 14 tables from the Kestrel extract and runs the share check from section 8.4: for
every batch the shares in the bridge add to 1.00 within a tolerance.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

con = k.connect()
ONE_ROW_IS = {
    "item": "a material, product or consumable", "vessel": "a tank, kettle or other holder of liquid",
    "calendar_day": "one date, with brew day and period", "recipe_version": "one version of one recipe",
    "spec": "one limit for one test on one item, valid for a period", "brew": "one trip through the brewhouse",
    "batch": "one fermenter fill", "vessel_event": "one thing that happened in one vessel",
    "lot": "one quantity of one thing with an ID", "sample": "one physical sample pulled",
    "movement": "quantity moving from one node to another", "package_unit": "one pallet",
    "result": "one test result on one sample", "brew_batch": "one brew's share of one batch",
}
for t in k.CORE:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:15s} {n:5d} rows   one row is {ONE_ROW_IS[t]}")

The share check. A batch whose shares add to 0.93 has a brew missing; no lineage query will find it.

In [ ]:
TOLERANCE = 0.001

def share_check(bridge):
    s = bridge.groupby("batch_key").share.sum().rename("share_sum").reset_index()
    s["ok"] = (s.share_sum - 1).abs() <= TOLERANCE
    return s

bridge = k.q(con, "SELECT * FROM brew_batch")
print(share_check(bridge).to_string(index=False))
print("with one brew row lost:")
print(share_check(bridge.drop(index=1)).to_string(index=False))

Things that change: vessel FV10 was replaced. Same code on the floor, two keys with valid dates.

In [ ]:
print(k.q(con, "SELECT vessel_key, vessel_code, valid_from, valid_to FROM vessel WHERE vessel_code = 'FV10'"))
print(k.q(con, "SELECT spec_key, test_code, low, high, version, valid_from, valid_to FROM spec "
               "WHERE item_key = 1060 AND test_code = 'IBU'"))

FV07 over one lager fill: fill to the end of CIP is the tank day figure for lager.

In [ ]:
ev = k.q(con, "SELECT event_type, start_utc, end_utc, batch_key, source FROM vessel_event WHERE vessel_key = "
              "(SELECT vessel_key FROM vessel WHERE vessel_code = 'FV07') ORDER BY start_utc")
print(ev.to_string(index=False))
days = (pd.Timestamp(ev[ev.event_type == "CIP"].end_utc.iloc[0]) - pd.Timestamp(ev.start_utc.iloc[0])).days
print(f"fill to end of CIP: {days} days")